In [1]:
import pandas as pd
import numpy as np

# 1. Cleaned Data Load karna
df = pd.read_csv("../data/processed/cleaned_superstore.csv")

# Ensure date columns are datetime type
df['order_date'] = pd.to_datetime(df['order_date'])
df['ship_date'] = pd.to_datetime(df['ship_date'])

print("Initial Shape:", df.shape)

# ==========================================
# 🎯 1. DATE & TIME FEATURES (For Time Series & Seasonality)
# ==========================================
df['order_year'] = df['order_date'].dt.year
df['order_month'] = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()
df['order_day'] = df['order_date'].dt.day
df['order_day_of_week'] = df['order_date'].dt.day_name()
df['order_quarter'] = df['order_date'].dt.quarter
df['is_weekend'] = df['order_date'].dt.dayofweek.isin([5, 6]).astype(int) # 1 if Sat/Sun else 0

# Yearly-Monthly period for time series aggregation (e.g., '2014-11')
df['year_month'] = df['order_date'].dt.to_period('M')


# ==========================================
# 💰 2. FINANCIAL & PROFITABILITY FEATURES
# ==========================================
# Unit Price (Single Item Price)
df['unit_price'] = df['sales'] / df['quantity']

# Profit Margin % (Kitna % profit margin hai order par)
# Profit Margin = (Profit / Sales) * 100
df['profit_margin_%'] = (df['profit'] / df['sales']) * 100

# Cost Price (Estimated total cost = Sales - Profit)
df['total_cost'] = df['sales'] - df['profit']

# Is Loss Making? (Binary Flag: 1 if loss, 0 if profit)
df['is_loss'] = (df['profit'] < 0).astype(int)


# ==========================================
# 📦 3. LOGISTICS & DISCOUNT FEATURES
# ==========================================
# Shipping Days (Delivery time)
df['shipping_days'] = (df['ship_date'] - df['order_date']).dt.days

# Discount Amount in Currency ($ / ₹)
# Original Price = Sales / (1 - Discount)
# Discount Amount = Original Price - Sales
df['discount_amount'] = np.where(
    df['discount'] > 0,
    (df['sales'] / (1 - df['discount'])) - df['sales'],
    0
)


# ==========================================
# 🔍 VERIFICATION & SUMMARY
# ==========================================
print("New Shape after Feature Engineering:", df.shape)
print("\nNewly Created Columns:")
new_cols = [
    'order_year', 'order_month', 'order_quarter', 'is_weekend', 
    'unit_price', 'profit_margin_%', 'total_cost', 'is_loss', 
    'shipping_days', 'discount_amount'
]
print(df[new_cols].head())

# Save dataset with Engineered Features for EDA & Modeling
df.to_csv("../data/processed/featured_superstore.csv", index=False)
print("\nSuccessfully saved engineered dataset to ../data/processed/featured_superstore.csv")

Initial Shape: (51290, 24)
New Shape after Feature Engineering: (51290, 37)

Newly Created Columns:
   order_year  order_month  order_quarter  is_weekend  unit_price  \
0        2014           11              4           0     110.990   
1        2014            2              1           0     412.155   
2        2014           10              4           0     575.019   
3        2014            1              1           0     578.502   
4        2014           11              4           0     354.120   

   profit_margin_%  total_cost  is_loss  shipping_days  discount_amount  
0        28.000000    159.8256        0              2            0.000  
1        -7.784693   3998.1600        1              2          412.155  
2        17.776630   4255.2000        0              1          575.019  
3        -3.337586   2989.0500        1              2          321.390  
4        10.996272   2521.4400        0              1            0.000  

Successfully saved engineered dataset to

After the feature engineering the dataset is saved in data/processed/featured_superstore.csv 
In this we have created several new columns just for the ease for the model making 